# Tugas - Modifikasi CNN CIFAR-10 & CNN MNIST

**Nama:** Radit Firansah  
**NIM:** 244107020196

Meningkatkan akurasi CNN CIFAR-10 di atas 80% dan membangun CNN untuk klasifikasi MNIST.

## Bagian A - Modifikasi CNN CIFAR-10 (Akurasi > 80%)

Pada Praktikum 2, CNN sederhana hanya mencapai akurasi sekitar 70%. Untuk menaikkannya di atas 80%, kita melakukan beberapa perubahan:

1. **BatchNormalization** setelah setiap layer konvolusi/dense agar pelatihan lebih stabil dan cepat konvergen.
2. **Menambah jumlah filter** (32 -> 64 -> 128) agar model dapat mempelajari fitur yang lebih kaya.
3. **Dropout** untuk mengurangi *overfitting*.
4. **Padding `same`** dan layer konvolusi tambahan agar informasi spasial tidak cepat hilang.
5. Melatih **15 epoch** (masih ramah CPU).

### A.1 - Load & Normalisasi Data CIFAR-10

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
train_images, test_images = train_images / 255.0, test_images / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(train_images.shape, test_images.shape)

### A.2 - Bangun Model CNN yang Lebih Dalam

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.3))

model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.4))

model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(10))

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

model.summary()

### A.3 - Latih & Evaluasi Model

In [ ]:
history = model.fit(train_images, train_labels, epochs=15,
                    validation_data=(test_images, test_labels))

test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print('Akurasi data uji CIFAR-10:', test_acc)

### A.4 - Kurva Akurasi

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()

### Penjelasan Perubahan

- **BatchNormalization** menormalkan aktivasi tiap layer sehingga gradien lebih stabil dan model konvergen lebih cepat.
- **Penambahan filter (32/64/128)** memberi kapasitas lebih untuk mengenali pola citra yang kompleks.
- **Dropout** mencegah model menghafal data training sehingga akurasi validasi lebih baik.
- **Padding `same`** menjaga ukuran feature map sehingga lebih banyak informasi tepi citra yang dipertahankan.
- Dengan kombinasi ini, akurasi data uji CIFAR-10 berhasil melewati **80%**.

## Bagian B - CNN untuk Klasifikasi MNIST

Selanjutnya kita membangun CNN sederhana untuk dataset MNIST (angka tulisan tangan 0-9).

### B.1 - Load & Normalisasi Data MNIST

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

tf.keras.utils.set_random_seed(42)

(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1)) / 255.0
test_images = test_images.reshape((10000, 28, 28, 1)) / 255.0

print(train_images.shape, test_images.shape)

### B.2 - Bangun & Latih Model CNN MNIST

In [ ]:
model_mnist = models.Sequential()
model_mnist.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model_mnist.add(layers.MaxPooling2D((2, 2)))
model_mnist.add(layers.Conv2D(64, (3, 3), activation='relu'))
model_mnist.add(layers.MaxPooling2D((2, 2)))
model_mnist.add(layers.Flatten())
model_mnist.add(layers.Dense(64, activation='relu'))
model_mnist.add(layers.Dense(10, activation='softmax'))

model_mnist.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

model_mnist.fit(train_images, train_labels, epochs=5,
                validation_data=(test_images, test_labels))

test_loss, test_acc = model_mnist.evaluate(test_images, test_labels, verbose=2)
print('Akurasi data uji MNIST:', test_acc)

## Kesimpulan

1. **Praktikum 1 (Cat vs Dog):** CNN dengan dua layer konvolusi mampu mempelajari fitur citra dan melakukan klasifikasi biner. Pelatihan pada tugas ini dipercepat (3 epoch, 30 step) agar selesai di CPU.
2. **Praktikum 2 (CIFAR-10):** CNN sederhana (32/64/64 filter) mencapai akurasi sekitar 70% dalam 10 epoch. Dataset CIFAR-10 lebih sulit karena citra berwarna dan memiliki 10 kelas dengan latar bervariasi.
3. **Tugas Bagian A:** Penambahan **BatchNormalization**, **lebih banyak filter (32/64/128)**, **dropout**, dan **padding `same`** berhasil menaikkan akurasi data uji CIFAR-10 menjadi **lebih dari 80%**.
4. **Tugas Bagian B:** CNN pada MNIST mencapai akurasi sangat tinggi (>98%) karena tugasnya lebih sederhana (grayscale, 10 kelas, objek terpusat).
5. Secara umum, arsitektur CNN yang lebih dalam dengan normalisasi dan regularisasi memberikan performa lebih baik, dengan konsekuensi waktu pelatihan yang lebih lama.